# HOG Ödevi Analiz Defteri

Bu notebook, **HOG tabanlı insan tespiti**, **özel nesne tespiti** ve **HOG + sınıflandırma** problemleri için yapılan çalışmaları özetlemek ve bazı görsel sonuçları incelemek için hazırlanmıştır.

Proje yapısı kısaca:

- `src/object_detection.py`  → OpenCV hazır HOG + SVM ile insan tespiti
- `src/custom_detection.py`  → Kendi eğittiğimiz HOG + LinearSVM ile özel nesne tespiti
- `src/classification.py`    → HOG feature + farklı sınıflandırıcılar (Linear SVM, RBF SVM, k-NN)
- `veri/pedestrians_results` → İnsan tespiti çıktı görüntüleri
- `veri/custom/results`      → Özel nesne tespiti çıktı görüntüleri
- `veri/custom/train_pos`    → Pozitif örnekler (nesne var)
- `veri/custom/train_neg`    → Negatif örnekler (nesne yok)


## 1. Proje Kök Dizinini ve Yol Yapısını Tanımlama

Bu hücrede, proje kök dizinini ve önemli klasörlerin yollarını tanımlıyoruz. Notebook genelde `notebooks/` klasörü içinde çalıştırıldığı için, bir üst dizine çıkıp proje kökünü buluyoruz.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Notebook genelde 'notebooks' klasöründen çalışacak
# Bir üst dizine çıkarak proje kökünü buluyoruz
NB_DIR = os.getcwd()
BASE_DIR = os.path.dirname(NB_DIR)

print("Notebook dizini  :", NB_DIR)
print("Proje kök dizini :", BASE_DIR)

# Önemli klasörler
PED_RESULTS_DIR = os.path.join(BASE_DIR, "veri", "pedestrians_results")
CUSTOM_RESULTS_DIR = os.path.join(BASE_DIR, "veri", "custom", "results")

print("Pedestrian sonuç klasörü:", PED_RESULTS_DIR)
print("Custom nesne sonuç klasörü:", CUSTOM_RESULTS_DIR)

## 2. Problem 1 – İnsan (Yaya) Tespiti Sonuçlarından Örnek

Bu bölümde `object_detection.py` tarafından üretilen sonuç görüntülerinden birini yükleyip gösteriyoruz.

Kod tarafında kullanılan yöntem:
- OpenCV'nin hazır `HOGDescriptor_getDefaultPeopleDetector()` modeli
- Multi-scale `detectMultiScale` ile farklı ölçeklerde tarama
- Non-Maximum Suppression (NMS) ile çakışan kutuları birleştirme
- Düşük skorlu tespitleri filtrelemek için `min_confidence` ve `hit_threshold`

Burada sadece görsel çıktıyı incelemek için bir örnek görsel gösteriyoruz.

In [ ]:
# Pedestrian sonuç klasöründen örnek bir görüntü göster
if os.path.exists(PED_RESULTS_DIR):
    files = [f for f in os.listdir(PED_RESULTS_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
    if len(files) > 0:
        sample_name = sorted(files)[0]
        sample_path = os.path.join(PED_RESULTS_DIR, sample_name)
        img = cv2.imread(sample_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6, 6))
        plt.imshow(img_rgb)
        plt.title(f"İnsan tespiti örneği: {sample_name}")
        plt.axis("off")
        plt.show()
    else:
        print("Pedestrian sonuç klasöründe görüntü bulunamadı.")
else:
    print("Pedestrian sonuç klasörü yok:", PED_RESULTS_DIR)

### 2.1. Threshold Değerlerinin Etkisi (Yorum)

Kod tarafında, insan tespiti için:

- `hit_threshold` (HOG + SVM dedektör eşiği)
- `min_confidence` (NMS öncesi düşük skorlu tespitleri eleme)

gibi parametrelerle oynanmıştır.

Yapılan gözlemler:

- **Düşük eşik (örneğin 0.0)**: Çok sayıda tespit üretilir, false positive sayısı artar.
- **Orta seviye eşik (örneğin 0.4–0.6)**: Gereksiz kutular azalır, insanlar yine yakalanır; genelde en dengeli sonuçlar.
- **Yüksek eşik (örneğin 1.0 ve üzeri)**: False positive belirgin şekilde azalır, fakat bazı insanlar hiç tespit edilemeyebilir (false negative artar).

Bu trade-off, raporda "threshold analiz" kısmında tartışılmıştır.

## 3. Problem 2 – Özel Nesne Tespiti Sonuçlarından Örnek

Bu bölümde `custom_detection.py` tarafından eğitilen HOG + Linear SVM modeliyle yapılan **özel nesne tespiti** sonuçlarından bir örnek gösteriyoruz.

Yöntem özetle:
- `train_pos` klasöründen pozitif örnekler (nesne var)
- `train_neg` klasöründen negatif örnekler (nesne yok)
- Bu patch'ler üzerinden HOG feature çıkarma
- Linear SVM ile 2 sınıflı sınıflandırıcı eğitme
- Test görüntüleri üzerinde sliding window + multi-scale tarama
- SVM skoruna göre pencereyi pozitif/negatif etiketleme
- NMS ile çakışan tespit kutularını birleştirme


In [ ]:
# Custom nesne sonuç klasöründen örnek bir görüntü göster
if os.path.exists(CUSTOM_RESULTS_DIR):
    files = [f for f in os.listdir(CUSTOM_RESULTS_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
    if len(files) > 0:
        sample_name = sorted(files)[0]
        sample_path = os.path.join(CUSTOM_RESULTS_DIR, sample_name)
        img = cv2.imread(sample_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6, 6))
        plt.imshow(img_rgb)
        plt.title(f"Özel nesne tespiti örneği: {sample_name}")
        plt.axis("off")
        plt.show()
    else:
        print("Custom sonuç klasöründe görüntü bulunamadı.")
else:
    print("Custom sonuç klasörü yok:", CUSTOM_RESULTS_DIR)

### 3.1. Karar Eşiği ve NMS Etkisi (Yorum)

`custom_detection.py` içinde aşağıdaki parametreler önemli rol oynamaktadır:

- `DECISION_THRESHOLD` : SVM decision function çıktı eşiği. Artırıldığında model daha seçici olur, tespit sayısı azalır.
- `NMS_IOU_THRESH` : NMS sırasında kullanılan IoU (Intersection over Union) eşiği. Küçük değerler daha agresif birleştirme yapar.

Gözlemler:

- Düşük `DECISION_THRESHOLD` (örneğin 0.0): Çok fazla tespit (özellikle kalabalık sahnelerde onlarca/hundreds kutu), false positive oranı yüksek.
- Orta-yüksek `DECISION_THRESHOLD` (örneğin 1.0 civarı): Gereksiz kutular azalır, sadece daha yüksek skorlu pencereler kalır.
- `NMS_IOU_THRESH` değerini 0.3 → 0.2 gibi düşürmek, üst üste binen kutuları tek kutuya indirerek görüntüyü sadeleştirir.


## 4. Problem 3 – Sınıflandırma (HOG + Farklı Modeller)

`classification.py` dosyasında, HOG feature'ları kullanarak aşağıdaki sınıflandırıcılar eğitildi:

- Linear SVM
- RBF çekirdekli SVM (RBF SVM)
- k-En Yakın Komşu (k-NN, k=5)

Veri seti:
- Pozitif örnekler: `veri/custom/train_pos` klasörü
- Negatif örnekler: `veri/custom/train_neg` klasörü
- Tüm HOG feature'lar birleştirilip, %70 eğitim / %30 test olacak şekilde bölündü (`train_test_split`).

Her model için konsolda şunlar raporlandı:

- Accuracy (doğruluk)
- Confusion matrix
- Precision / Recall / F1-score değerlerini içeren classification report

Notebook içinde ayrıca, kaydedilen en iyi modeli `hog_classifier.joblib` dosyasından yükleyip tek bir görüntü üzerinde tahmin yapılabilir.

In [ ]:
from pathlib import Path

CLASSIFIER_PATH = os.path.join(CUSTOM_DIR, "hog_classifier.joblib")
print("Classifier dosyası yolu:", CLASSIFIER_PATH)

if os.path.exists(CLASSIFIER_PATH):
    clf = joblib.load(CLASSIFIER_PATH)
    print("Model yüklendi:", type(clf))
else:
    print("Uyarı: hog_classifier.joblib bulunamadı. Önce classification.py çalıştırılmalı.")

### 4.1. Örnek Tahmin (İsteğe Bağlı)

Aşağıdaki hücre, eğer `hog_classifier.joblib` oluşturulduysa, `train_pos` klasöründen bir örnek üzerinde tahmin yapar.

> Not: Bu hücre tamamen isteğe bağlıdır. Çalıştırılmasa bile, ödev için gerekli klasör ve notebook yapısı sağlanmış olur.

In [ ]:
TRAIN_POS_DIR = os.path.join(CUSTOM_DIR, "train_pos")

def get_hog_descriptor():
    return cv2.HOGDescriptor(
        _winSize=(64, 128),
        _blockSize=(16, 16),
        _blockStride=(8, 8),
        _cellSize=(8, 8),
        _nbins=9,
    )

if os.path.exists(CLASSIFIER_PATH) and os.path.exists(TRAIN_POS_DIR):
    clf = joblib.load(CLASSIFIER_PATH)
    hog = get_hog_descriptor()

    files = [f for f in os.listdir(TRAIN_POS_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
    if len(files) > 0:
        sample_name = sorted(files)[0]
        sample_path = os.path.join(TRAIN_POS_DIR, sample_name)

        img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
        img_resized = cv2.resize(img, (64, 128))
        hog_desc = hog.compute(img_resized).flatten().reshape(1, -1)

        pred = clf.predict(hog_desc)[0]

        plt.imshow(img_resized, cmap="gray")
        plt.title(f"Örnek pozitif patch, tahmin sınıfı: {pred}")
        plt.axis("off")
        plt.show()
    else:
        print("train_pos klasöründe görüntü bulunamadı.")
else:
    print("Classifier veya train_pos dizini bulunamadı. Bu hücre isteğe bağlıdır.")

## 5. Sonuç

Bu notebook ile:

- HOG tabanlı **insan tespiti** sonuçlarından örnekler görüntülendi ve threshold parametrelerinin etkisi tartışıldı.
- **Özel nesne tespiti** için eğitilen HOG + LinearSVM modelinin sonuçları incelendi.
- HOG feature'ları ile eğitilen farklı sınıflandırıcılar (Linear SVM, RBF SVM, k-NN) özetlendi ve en iyi model `hog_classifier.joblib` olarak kaydedildi.

Bu defter, ödev raporundaki anlatımı destekleyen ek bir analiz ve görselleştirme aracı olarak kullanılmıştır.